In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt


In [ ]:

# Open in read-only mode
root = zarr.open_group("/dtu-compute/msaca/sliceA_xray_pc/sliceA_reconstructions.ome.zarr", mode="r")

# Access the full-resolution arrays (level 0)
xray_lvl0 = root["xray/0"]
xray_lvl1 = root["xray/1"]
neutron_lvl0 = root["neutron/0"]

print("X-ray shape:", xray_lvl0.shape)
print("Neutron shape:", neutron_lvl0.shape)

# Example: load a subvolume
# Let's say you want z=100:200, y=500:800, x=400:700
sub_xray_big = xray_lvl0[1600:2400, 300:400, 1400:2000]
sub_xray = xray_lvl1[800:1200, 150:200, 700:1000]  # note: different resolution
sub_neutron = neutron_lvl0[800:1200, 150:200, 700:1000]  # note: different resolution

print("Sub X-ray shape:", sub_xray.shape)
print("Sub Neutron shape:", sub_neutron.shape)

In [ ]:
plt.imshow(sub_xray_big[:,0])
plt.colorbar()
plt.show()
plt.imshow(sub_neutron[:,0])
plt.colorbar()
plt.show()

In [ ]:
import zarr
import SimpleITK as sitk
import numpy as np

# --------------------------
# Step 1: open the OME-Zarr datasets
# --------------------------
levelx = 2   # x-ray pyramid level (4× downsampled)
leveln = 1   # neutron pyramid level (2× downsampled)

root = zarr.open_group("/dtu-compute/msaca/sliceA_xray_pc/sliceA_reconstructions.ome.zarr", mode="r")
xray = root[f"xray/{levelx}"]
neutron = root[f"neutron/{leveln}"]

# correct spacings
spacing_xray = [3.1*(2**levelx)] * 3
spacing_neutron = [6.2*(2**leveln)] * 3

# correct coordinate scaling
coords_xray = [
    (2600//(2**levelx),  250//(2**levelx), 1000//(2**levelx)),
    (600//(2**levelx),   400//(2**levelx), 2400//(2**levelx)),
    (1400//(2**levelx),  300//(2**levelx), 2500//(2**levelx)),
]
coords_neutron = [
    (1300//(2**leveln), 125//(2**leveln),  500//(2**leveln)),
    (300//(2**leveln),  200//(2**leveln), 1200//(2**leveln)),
    (700//(2**leveln),  150//(2**leveln), 1250//(2**leveln)),
]
size = (64, 64, 64)  # use 128³ subvolumes

def extract_subvolume_with_origin(zarr_array, start, size, spacing):
    z0, y0, x0 = start
    dz, dy, dx = size
    arr = zarr_array[z0:z0+dz, y0:y0+dy, x0:x0+dx].astype(np.float32)
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(spacing)
    origin = [x0*spacing[2], y0*spacing[1], z0*spacing[0]]
    img.SetOrigin(origin)
    img.SetDirection(np.eye(3).flatten())
    return img

# Extract and track correct origins
xray_subs = [extract_subvolume_with_origin(xray, s, size, spacing_xray) for s in coords_xray]
neutron_subs = [extract_subvolume_with_origin(neutron, s, size, spacing_neutron) for s in coords_neutron]


In [ ]:
for i, (fx, mv) in enumerate(zip(neutron_subs, xray_subs)):
    print(f"Pair {i}")
    print(" Fixed size:", fx.GetSize(), "spacing:", fx.GetSpacing(), "origin:", fx.GetOrigin())
    print(" Moving size:", mv.GetSize(), "spacing:", mv.GetSpacing(), "origin:", mv.GetOrigin())


In [ ]:

# --------------------------
# Step 4: registration framework
# --------------------------
def register_pair_similarity(fixed, moving, initial=None):
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMeanSquares()
    reg.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.0,
        minStep=1e-4,
        numberOfIterations=200
    )
    reg.SetInterpolator(sitk.sitkLinear)

    if initial is None:
        initial = sitk.Similarity3DTransform()
        # Initialize at the geometric center of the fixed & moving images
        initial = sitk.CenteredTransformInitializer(
            fixed,
            moving,
            initial,
            sitk.CenteredTransformInitializerFilter.GEOMETRY
        )

    reg.SetInitialTransform(initial, inPlace=False)

    final_transform = reg.Execute(fixed, moving)
    return final_transform


local_transforms = [register_pair_similarity(f, m) for f, m in zip(neutron_subs, xray_subs)]

# --------------------------
# Step 5: build a global Similarity3DTransform
# --------------------------
# helpers -------------------------------------------------------------
def centers_from_coords(coords, spacing, size):
    # coords are (z,y,x); spacing is [sz, sy, sx]; size is (dz,dy,dx)
    out = []
    for (z,y,x) in coords:
        cx = (x + size[2]/2) * spacing[2]
        cy = (y + size[1]/2) * spacing[1]
        cz = (z + size[0]/2) * spacing[0]
        out.append(np.array([cx, cy, cz]))
    return out

def fit_similarity_3d(fixed_pts, moving_pts):
    fixed_pts = np.asarray(fixed_pts)
    moving_pts = np.asarray(moving_pts)
    cF, cM = fixed_pts.mean(0), moving_pts.mean(0)
    F, M = fixed_pts - cF, moving_pts - cM
    H = M.T @ F
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = Vt.T @ U.T
    scale = np.sum(S) / np.sum(M**2)
    T = cF - scale * (R @ cM)
    tr = sitk.Similarity3DTransform()
    tr.SetMatrix((scale * R).flatten())
    tr.SetTranslation(tuple(T))
    return tr

# centers in moving (xray) and fixed (neutron) spaces -----------------
centers_moving  = centers_from_coords(coords_xray,    spacing_xray,    size)
centers_fixed   = centers_from_coords(coords_neutron, spacing_neutron, size)

# map each moving-center through its local transform to fixed space ----
centers_mapped = [
    np.array(local_T.TransformPoint(tuple(cm)))
    for local_T, cm in zip(local_transforms, centers_moving)
]

# fit global similarity: moving -> fixed -------------------------------
global_similarity = fit_similarity_3d(fixed_pts=centers_mapped,
                                      moving_pts=centers_moving)


In [ ]:
# global_similarity is a sitk.Similarity3DTransform

# Translation vector
translation = global_similarity.GetTranslation()

# 3x3 rotation+scale matrix (flattened in row-major order)
matrix = np.array(global_similarity.GetMatrix()).reshape(3, 3)

# Uniform scale factor
scale = global_similarity.GetScale()

# Pure rotation matrix (matrix / scale)
rotation = matrix / scale

print("=== Global Similarity Transform Parameters ===")
print("Translation (x,y,z):", translation)
print("Scale:", scale)
print("Rotation matrix:")
print(rotation)


In [ ]:

# --------------------------
# Step 6: apply to X-ray, keeping its own resolution
# --------------------------
img_xray = sitk.GetImageFromArray(xray[:].astype(np.float32))
img_xray.SetSpacing(spacing_xray)
img_xray.SetOrigin((0, 0, 0))

reference = sitk.Image(img_xray.GetSize(), sitk.sitkFloat32)
reference.CopyInformation(img_xray)

xray_registered = sitk.Resample(
    img_xray,
    reference,
    global_similarity,
    sitk.sitkLinear,
    0.0,
    sitk.sitkFloat32
)

In [ ]:
xreg = sitk.GetArrayFromImage(xray_registered)   # registered x-ray
xraw = sitk.GetArrayFromImage(img_xray)          # original x-ray
nraw = sitk.GetArrayFromImage(
    sitk.GetImageFromArray(neutron[:].astype(np.float32))
)                                                # neutron

# pick a slice index along y-axis
yidx = 130

# define zoom region in z and x (rows, cols)
zmin, zmax = 800, 1000   # adjust as needed
xmin, xmax = 300, 500

# crop for display
xreg_crop = xreg[zmin:zmax, yidx, xmin:xmax]
xraw_crop = xraw[zmin:zmax, yidx, xmin:xmax]
nraw_crop = nraw[zmin:zmax, yidx, xmin:xmax]

# plot side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(xreg_crop, cmap='gray')
axes[0].set_title("X-ray Registered")

axes[1].imshow(xraw_crop, cmap='gray')
axes[1].set_title("X-ray Original")

axes[2].imshow(nraw_crop, cmap='gray')
axes[2].set_title("Neutron")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()
